# Module 8b: The hypothesis-testing toolkit

**Goal:** choose, run, check and report the **right test for your design**, and be able to explain why. Module 8 gave the logic; this module gives the toolkit a thesis needs.

**You will learn:** the anatomy of a test (statistic → distribution → p-value) · one-sample, two-sample and **paired** *t*-tests · rank-based (non-parametric) tests ·
**ANOVA** and post-hoc comparisons · using a baseline (**gain scores and ANCOVA**) · tests for proportions and tables · correlation tests ·
**equivalence** testing · **power and sample size** · charts for inference · a decision guide and reporting templates

**Maths in this module:** the $t$, $F$, $\chi^2$ and normal statistics · degrees of freedom · ranks and the $U$ statistic · sums of squares · Fisher's $z$ · effect sizes (Cohen's $d$, $\eta^2$, odds ratio, Cramér's $V$) · power

> 🧭 Always the same workflow: **(1) state the question and the design → (2) pick the estimand (what quantity?) → (3) check assumptions with plots → (4) run the test *and* get an interval and an effect size → (5) report plainly.**
> Decide steps 1–3 **before** looking at the results.

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import ttost_ind

sns.set_theme(style="whitegrid", palette="colorblind")
trial = pd.read_csv(DATA / "app_trial.csv")
students = pd.read_csv(DATA / "students.csv")
trial["gain"] = trial["post_score"] - trial["pre_score"]
by = {g: trial[trial["group"] == g] for g in ("control", "tutorial", "gamified")}
trial.head()

## The data: a randomised app experiment

`app_trial.csv` has 300 users randomly assigned to **control**, **tutorial** or **gamified** onboarding (100 each). For each user: a `pre_score` and `post_score` (a skill test before and after), `minutes_active` (skewed), `converted` (0/1),
a 1–5 `rating` (ordinal) and `device`. Different outcome types need different tests, and that's the point of this module.

## 1. Anatomy of a test, by hand

Every test has the same skeleton: **test statistic** (how far is the data from $H_0$, in standard-error units?) → its **distribution if $H_0$ is true** → the **p-value** (tail area beyond what we saw).

**One-sample $t$-test.** Is the mean exam score different from a benchmark $\mu_0=65$?

$$t=\frac{\bar x-\mu_0}{s/\sqrt n},\qquad \text{df}=n-1,\qquad p=2\,P(T_{n-1}\ge|t|)\ \text{(two-sided)}$$

In [ ]:
x = students["exam_score"].to_numpy()
mu0, n = 65, len(x)
se = x.std(ddof=1) / math.sqrt(n)
t_stat = (x.mean() - mu0) / se
p_two = 2 * stats.t.sf(abs(t_stat), df=n - 1)
ci = stats.t.interval(0.95, df=n - 1, loc=x.mean(), scale=se)

print(f"t = {t_stat:.3f}, df = {n - 1}, two-sided p = {p_two:.5f}, 95% CI for the mean {ci[0]:.2f} to {ci[1]:.2f}")
print("scipy:", stats.ttest_1samp(x, mu0))
print(f"Cohen's d = {(x.mean() - mu0) / x.std(ddof=1):.2f}")

**Two things to notice**

* **Test and interval agree.** The 95% CI (66.5 to 69.4) excludes 65, and the test rejects $\mu=65$ at 5%. A test at level $\alpha$ rejects exactly when the $(1-\alpha)$ interval excludes the null value. **The interval tells you more** (how far from 65?).
* **One-sided vs two-sided.** A one-sided test (`alternative="greater"`) puts all $\alpha$ in one tail, halving $p$. It is legitimate **only if chosen in advance** because the opposite direction would be meaningless or irrelevant. Otherwise use two-sided, the default expectation in a thesis.

## 2. Two independent groups

**Welch's $t$-test** (does not assume equal variances) is the safe default:

$$t=\frac{\bar x_1-\bar x_2}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$$

The classic **Student's** version pools the variances; it matches Welch when the groups have equal size and spread, and misleads when they don't. **Levene's test** checks equality of variances (but see the caution on pre-testing in §9).
Question: did **gamified** users score higher on `post_score` than **control** users?

In [ ]:
gam, ctl = by["gamified"]["post_score"], by["control"]["post_score"]

welch = stats.ttest_ind(gam, ctl, equal_var=False)
low, high = welch.confidence_interval(0.95)
print(f"difference in means: {gam.mean() - ctl.mean():.2f} points (95% CI {low:.2f} to {high:.2f})")
print(f"Welch t = {welch.statistic:.2f}, df = {welch.df:.1f}, p = {welch.pvalue:.4f}")
print(f"Student's t p = {stats.ttest_ind(gam, ctl).pvalue:.4f}   (nearly identical: similar variances; Levene p = {stats.levene(gam, ctl).pvalue:.2f})")

sp = math.sqrt(((len(gam) - 1) * gam.var() + (len(ctl) - 1) * ctl.var()) / (len(gam) + len(ctl) - 2))
d = (gam.mean() - ctl.mean()) / sp
hedges_g = d * (1 - 3 / (4 * (len(gam) + len(ctl)) - 9))
print(f"Cohen's d = {d:.2f}, Hedges' g = {hedges_g:.2f}  (a small-to-medium effect)")

**Effect size** $d=\dfrac{\bar x_1-\bar x_2}{s_p}$ (pooled sd); **Hedges' $g$** applies a small-sample correction. Always report the difference *in original units* with its interval **and** a standardised effect size.

## 3. Paired designs: the same people measured twice

When each person is measured **before and after** (or two matched conditions), analyse the **within-person differences** $d_i=\text{after}_i-\text{before}_i$ with a one-sample $t$-test on those differences (**paired $t$-test**):

$$t=\frac{\bar d}{s_d/\sqrt n},\quad\text{df}=n-1,\qquad d_z=\frac{\bar d}{s_d}$$

Pairing removes the large, stable differences *between* people, so it is usually far more powerful. Using the 100 **tutorial** users:

In [ ]:
tut = by["tutorial"]
paired = stats.ttest_rel(tut["post_score"], tut["pre_score"])
diff = tut["post_score"] - tut["pre_score"]
ci_p = paired.confidence_interval(0.95)
print(f"mean improvement {diff.mean():.2f} points (95% CI {ci_p.low:.2f} to {ci_p.high:.2f}); paired t = {paired.statistic:.2f}, p = {paired.pvalue:.1e}; d_z = {diff.mean() / diff.std(ddof=1):.2f}")

wrong = stats.ttest_ind(tut["post_score"], tut["pre_score"], equal_var=False)
print(f"treating the same people as two independent groups (WRONG): p = {wrong.pvalue:.4f}")
print(f"correlation between each person's before and after scores: r = {tut['pre_score'].corr(tut['post_score']):.2f}")
print("non-parametric paired test (Wilcoxon signed-rank): p =", f"{stats.wilcoxon(diff).pvalue:.1e}")

The paired $p$ is about $10^{-11}$; the (incorrect) independent-samples test gives only $0.002$. Before and after scores correlate at 0.84 within this group, so most of the variation is stable *between* people and cancels in the differences.
**Using the wrong test for the design is one of the most common errors in student theses.** The **Wilcoxon signed-rank test** is the distribution-free alternative for paired data.

## 4. Rank-based tests: skewed and ordinal outcomes

`minutes_active` is strongly right-skewed (Module 7b). **Mann–Whitney $U$** (Wilcoxon rank-sum) replaces values by their **ranks** (1 = smallest overall) and asks whether one group tends to have higher ranks.

$$U_1=R_1-\frac{n_1(n_1+1)}{2}\quad(R_1=\text{sum of group 1's ranks}),\qquad \text{probability of superiority}=\frac{U_1}{n_1n_2}$$

The last quantity is the chance that a random user from group 1 has a larger value than a random user from group 2: an effect size that is easy to explain (0.5 = no difference).

In [ ]:
mg, mc = by["gamified"]["minutes_active"].to_numpy(), by["control"]["minutes_active"].to_numpy()

ranks = stats.rankdata(np.r_[mg, mc])
U1 = ranks[: len(mg)].sum() - len(mg) * (len(mg) + 1) / 2
res = stats.mannwhitneyu(mg, mc, alternative="two-sided")
print(f"U by ranks = {U1:.0f}; scipy U = {res.statistic:.0f}; p = {res.pvalue:.4f}")
print(f"probability of superiority = {U1 / (len(mg) * len(mc)):.3f}  (a random gamified user is more active than a random control user {U1 / (len(mg) * len(mc)):.0%} of the time)")
print(f"for comparison, Welch's t on the raw skewed minutes: p = {stats.ttest_ind(mg, mc, equal_var=False).pvalue:.4f}")
print(f"3-group Kruskal-Wallis on minutes: p = {stats.kruskal(*[by[g]['minutes_active'] for g in by]).pvalue:.4f}")
print(f"ordinal 1-5 rating, gamified vs control (Mann-Whitney): p = {stats.mannwhitneyu(by['gamified']['rating'], by['control']['rating']).pvalue:.1e}")

**When to use rank tests:** ordinal data (ratings), strongly skewed or heavy-tailed outcomes, small samples where normality is doubtful, or data with outliers you can't justify removing.
**Caveats:** (1) the $U$ test compares whole distributions (stochastic dominance); it is a test of **medians only if the two distributions have the same shape**; (2) with large samples and mild skew, the $t$-test is usually fine; (3) a **log transform** (Module 7b) plus a $t$-test is a good alternative that gives interpretable ratios.

## 5. Three or more groups: one-way ANOVA

ANOVA asks whether *any* group mean differs, by comparing **between-group** to **within-group** variation:

$$F=\frac{\text{MSB}}{\text{MSW}}=\frac{\text{SSB}/(k-1)}{\text{SSW}/(N-k)},\qquad \eta^2=\frac{\text{SSB}}{\text{SSB}+\text{SSW}}$$

where SSB $=\sum n_j(\bar x_j-\bar x)^2$ and SSW $=\sum\sum(x_{ij}-\bar x_j)^2$.

In [ ]:
groups = [by[g]["post_score"].to_numpy() for g in ("control", "tutorial", "gamified")]
all_values = np.concatenate(groups)
grand, k, N = all_values.mean(), len(groups), len(all_values)

ssb = sum(len(v) * (v.mean() - grand) ** 2 for v in groups)
ssw = sum(((v - v.mean()) ** 2).sum() for v in groups)
F = (ssb / (k - 1)) / (ssw / (N - k))
p = stats.f.sf(F, k - 1, N - k)
print(f"F({k - 1}, {N - k}) = {F:.2f}, p = {p:.4f}, eta^2 = {ssb / (ssb + ssw):.3f}   (scipy: {stats.f_oneway(*groups)})")
print(f"Levene's test for equal variances: p = {stats.levene(*groups).pvalue:.2f}")

The overall test is **not significant** ($p=0.085$), and the effect is small ($\eta^2=0.016$: group explains 1.6% of the variance in post scores).
Yet the comparison *gamified vs control* alone had $p=0.026$ in §2. How can both be true? That is the **multiple-comparisons** problem.

### Post-hoc comparisons and corrections
If ANOVA is significant, ask *which* groups differ, with a procedure that controls the family-wise error rate: **Tukey's HSD** for all pairwise differences. Alternatively, correct a set of $p$-values:
**Bonferroni** ($p\times m$, capped at 1), **Holm** (step-down; never worse than Bonferroni), or **Benjamini–Hochberg** (controls the false discovery rate; for many tests).

In [ ]:
print(pairwise_tukeyhsd(trial["post_score"], trial["group"]).summary())

pairs = [("tutorial", "control"), ("gamified", "control"), ("gamified", "tutorial")]
raw_p = [stats.ttest_ind(by[a]["post_score"], by[b]["post_score"], equal_var=False).pvalue for a, b in pairs]
table = pd.DataFrame({"comparison": [f"{a} vs {b}" for a, b in pairs], "raw p": raw_p})
for method in ("bonferroni", "holm", "fdr_bh"):
    table[method] = multipletests(raw_p, method=method)[1]
table.round(4)

After correcting for three comparisons, gamified vs control has $p\approx0.08$. **Which is "right"?** It depends on what you decided *in advance*:

* If **one** comparison (gamified vs control) was your pre-registered primary hypothesis, report its uncorrected $p$ (and say that the other comparisons were exploratory).
* If you looked at all three and report whichever is smallest, **correct** for three.

Write your comparisons down **before** analysing (your project proposal!). This is what separates a confirmatory from an exploratory analysis.

## 6. Use the baseline: gain scores and ANCOVA

We have a `pre_score` for everyone, and the groups were randomised, so we expect balanced baselines (checked formally in §9). Analysing only `post_score` ignores the biggest source of between-person variation. Two better analyses:

* **Gain scores:** analyse $\text{post}-\text{pre}$.
* **ANCOVA:** regress post on the group **and** the baseline: $\text{post}_i=\beta_0+\beta_1\text{pre}_i+\gamma_{\text{tutorial}}+\gamma_{\text{gamified}}+\varepsilon_i$. The $\gamma$'s are baseline-adjusted group effects, usually the most precise option.

In [ ]:
print(trial.groupby("group")["gain"].agg(["mean", "std", "count"]).round(2))
gains = [by[g]["gain"].to_numpy() for g in ("control", "tutorial", "gamified")]
print("\nANOVA on gain scores:", stats.f_oneway(*gains))

ancova = smf.ols("post_score ~ pre_score + C(group, Treatment('control'))", data=trial).fit()
coefs = pd.DataFrame({"coef": ancova.params, "ci_low": ancova.conf_int()[0], "ci_high": ancova.conf_int()[1], "p": ancova.pvalues}).round(3)
print("\nANCOVA coefficients (effects relative to control, adjusted for baseline):")
print(coefs)
print(f"\nresidual sd: ANCOVA {math.sqrt(ancova.mse_resid):.1f} vs about {trial.groupby('group')['post_score'].std().mean():.1f} when ignoring the baseline")

The **same data**, analysed properly, tell a clear story: gamified onboarding raised scores by about **4.4 points** (95% CI 2.7 to 6.1) over control, and tutorial by **2.4** (0.8 to 4.1); the ANOVA on gains has $p\approx2.5\times10^{-6}$.
The baseline soaks up variation, halving the residual standard deviation (about 12 → 6). **Design and analysis choices change power dramatically.** Decide yours in advance: *the baseline adjusted analysis (ANCOVA) is usually the right primary analysis for a randomised before/after study.*

## 7. Proportions and binary outcomes

`converted` is 0/1. With $x$ successes in $n$ trials the proportion is $\hat p=x/n$. Useful intervals:

* **Wilson interval** (good default, even for small $n$): `proportion_confint(x, n, method="wilson")`.
* **Two proportions:** $z=\dfrac{\hat p_1-\hat p_2}{\sqrt{\hat p(1-\hat p)(1/n_1+1/n_2)}}$ with the *pooled* $\hat p$ in the test; the CI for the difference uses the unpooled standard error.
* **Effect measures:** difference $\hat p_1-\hat p_2$; **risk ratio** $\hat p_1/\hat p_2$; **odds ratio** $\dfrac{\hat p_1/(1-\hat p_1)}{\hat p_2/(1-\hat p_2)}$.

In [ ]:
conv = {g: (int(by[g]["converted"].sum()), len(by[g])) for g in by}
print("conversions:", conv)

x1, n1 = conv["gamified"]; x0, n0 = conv["control"]
z, p = proportions_ztest([x1, x0], [n1, n0])
lo, hi = confint_proportions_2indep(x1, n1, x0, n0, method="wald")
p1, p0 = x1 / n1, x0 / n0
print(f"gamified {p1:.0%} vs control {p0:.0%}: difference {p1 - p0:.2f} (95% CI {lo:.2f} to {hi:.2f}); z = {z:.2f}, p = {p:.4f}")
print(f"risk ratio {p1 / p0:.2f}; odds ratio {(p1 / (1 - p1)) / (p0 / (1 - p0)):.2f}")
print("Wilson 95% CI for gamified conversion:", np.round(proportion_confint(x1, n1, method="wilson"), 3))

table3 = np.array([[conv[g][0], conv[g][1] - conv[g][0]] for g in by])
chi2, p3, dof, expected = stats.chi2_contingency(table3)
print(f"all three groups (chi-square test of homogeneity): chi2 = {chi2:.2f}, df = {dof}, p = {p3:.4f}")

**Fisher's exact test** is for small counts, where the chi-square approximation fails (a rule of thumb: some **expected count < 5**):

In [ ]:
small = np.array([[3, 7], [9, 1]])             # e.g. 3 of 10 treated improved vs 9 of 10 controls
odds_ratio, p_fisher = stats.fisher_exact(small)
print(f"Fisher exact: odds ratio = {odds_ratio:.3f}, p = {p_fisher:.4f}")
print("expected counts under independence:\n", stats.contingency.expected_freq(small).round(1))

## 8. Association between two categorical variables

The **chi-square test of independence** compares observed counts with those expected if the variables were unrelated: $E_{ij}=\dfrac{(\text{row}_i\text{ total})(\text{col}_j\text{ total})}{\text{grand total}}$, $\chi^2=\sum\frac{(O-E)^2}{E}$, $\text{df}=(r-1)(c-1)$.
**Cramér's $V$**$=\sqrt{\dfrac{\chi^2}{N(\min(r,c)-1)}}$ is the effect size (0 = none, 1 = perfect). Is `device` related to `group`? (It shouldn't be, because assignment was random.)

In [ ]:
observed = pd.crosstab(trial["group"], trial["device"])
row, col, total = observed.sum(axis=1).to_numpy(), observed.sum(axis=0).to_numpy(), observed.to_numpy().sum()
expected = np.outer(row, col) / total
chi2 = ((observed.to_numpy() - expected) ** 2 / expected).sum()
dof = (observed.shape[0] - 1) * (observed.shape[1] - 1)
v = math.sqrt(chi2 / (total * (min(observed.shape) - 1)))
print(observed)
print(f"\nchi2 = {chi2:.2f}, df = {dof}, p = {stats.chi2.sf(chi2, dof):.2f}, smallest expected count = {expected.min():.1f}, Cramer's V = {v:.3f}")

$p=0.77$, $V\approx0.06$: no evidence of imbalance in devices across groups, and all expected counts are well above 5, so the chi-square approximation is fine.

## 9. Correlation tests

* **Pearson** $r$ (linear association, continuous, roughly bivariate normal), **Spearman** $\rho$ (Pearson on ranks; monotonic relationships, ordinal data, outliers), **Kendall** $\tau$ (concordant vs discordant pairs; good for small $n$ with ties).
* Interval for $r$ via **Fisher's $z$**: $z=\operatorname{arctanh}(r)$ has standard error $1/\sqrt{n-3}$; compute the interval for $z$ and transform back with $\tanh$.

In [ ]:
r_result = stats.pearsonr(trial["pre_score"], trial["post_score"])
r, n = r_result.statistic, len(trial)
z = np.arctanh(r); se_z = 1 / math.sqrt(n - 3)
lo, hi = np.tanh(z - 1.96 * se_z), np.tanh(z + 1.96 * se_z)
print(f"Pearson r = {r:.3f} (95% CI {lo:.3f} to {hi:.3f}), p = {r_result.pvalue:.1e}")
print(f"Spearman rho = {stats.spearmanr(trial['pre_score'], trial['post_score'])[0]:.3f}, Kendall tau = {stats.kendalltau(trial['pre_score'], trial['post_score'])[0]:.3f}")

### A caution about pre-testing assumptions
A common routine is "run Shapiro–Wilk and Levene first, then pick the test". It's tempting but flawed: those tests have little power in small samples (they miss real problems) and too much in large ones (flag harmless ones), and a two-stage procedure changes the error rates.
A better habit: **choose the method from the design and data type, check assumptions with plots (histograms, Q-Q), prefer methods robust to violations (Welch, rank tests, permutation/bootstrap), and run a sensitivity analysis** (does the conclusion survive a second reasonable method?).
For gamified vs control on `post_score`, Welch ($p=0.026$) and Mann–Whitney ($p=0.029$) agree. That is what you want to be able to say.

## 10. "No difference" needs a different test: equivalence

A non-significant result is **not** evidence of equality. To argue two groups are *practically equivalent*, use **TOST** (two one-sided tests): pick an **equivalence margin** $\pm\Delta$ *in advance* (the largest difference you would consider unimportant) and reject both "difference $\le-\Delta$" and "difference $\ge+\Delta$".
A natural use: **checking baseline balance** in a randomised trial.

In [ ]:
pre_g, pre_c = by["gamified"]["pre_score"], by["control"]["pre_score"]
p_tost, lower_test, upper_test = ttost_ind(pre_g, pre_c, -5, 5)
print(f"baseline difference (gamified - control) = {pre_g.mean() - pre_c.mean():.2f} points")
print(f"TOST p = {p_tost:.4f} -> the groups' baselines are equivalent within +/- 5 points" if p_tost < 0.05 else f"TOST p = {p_tost:.3f}: equivalence not shown")

## 11. Power and sample size: plan before you collect

**Power** is the probability of detecting a real effect of a given size. It depends on the effect size, the sample size, $\alpha$ and the variability. Use it in two ways:
**a priori** (how many participants do I need?), and **minimum detectable effect** (what could I detect with the sample I can get?). Reviewers expect this in a thesis proposal.
*(Avoid "observed power" computed from your result: it's just a re-expression of the $p$-value.)*

In [ ]:
power = TTestIndPower()
n_needed = power.solve_power(effect_size=0.5, alpha=0.05, power=0.8)
p_100 = power.power(effect_size=0.3, nobs1=100, alpha=0.05)
mde_100 = power.solve_power(nobs1=100, alpha=0.05, power=0.8)
print(f"to detect d = 0.5 with 80% power: {math.ceil(n_needed)} per group")
print(f"with 100 per group and d = 0.3 the power is only {p_100:.0%}")
print(f"with 100 per group the smallest effect detectable with 80% power is d = {mde_100:.2f}")

ns = np.arange(5, 201)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for d in (0.2, 0.5, 0.8):
    ax.plot(ns, [power.power(d, n, 0.05) for n in ns], label=f"d = {d}")
ax.axhline(0.8, color="grey", linestyle=":")
ax.set_xlabel("Participants per group"); ax.set_ylabel("Power"); ax.set_title("Small effects need far larger samples"); ax.legend()
plt.show()

### What p-values look like (a simulation worth seeing)
If $H_0$ is true, p-values are **uniformly distributed** (so 5% fall below 0.05 whatever the sample size). If there is a real effect, they pile up near 0, and power is the fraction below $\alpha$.

In [ ]:
rng = np.random.default_rng(0)
p_null = np.array([stats.ttest_ind(rng.normal(size=30), rng.normal(size=30)).pvalue for _ in range(2000)])
p_alt = np.array([stats.ttest_ind(rng.normal(size=30), rng.normal(0.6, 1, 30)).pvalue for _ in range(2000)])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
axes[0].hist(p_null, bins=20, edgecolor="white"); axes[0].set_title(f"No real effect: flat; {np.mean(p_null < 0.05):.1%} below 0.05")
axes[1].hist(p_alt, bins=20, edgecolor="white"); axes[1].set_title(f"Real effect d = 0.6, n = 30: {np.mean(p_alt < 0.05):.0%} below 0.05 (the power)")
for ax in axes: ax.set_xlabel("p-value")
plt.tight_layout(); plt.show()

## 12. Charts for inference: show intervals, not just bars

Two chart types every thesis uses: **means with 95% confidence intervals** (the interval is the star), and a **forest plot** comparing standardised effect sizes with their intervals across outcomes.
The standard error of Cohen's $d$ is approximately $\sqrt{\dfrac{n_1+n_2}{n_1n_2}+\dfrac{d^2}{2(n_1+n_2)}}$.

In [ ]:
def d_with_ci(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    n1, n2 = len(a), len(b)
    sp = math.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    d = (a.mean() - b.mean()) / sp
    se = math.sqrt((n1 + n2) / (n1 * n2) + d ** 2 / (2 * (n1 + n2)))
    return d, d - 1.96 * se, d + 1.96 * se

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

# (a) group means with 95% t-intervals, for the gain score
for i, g in enumerate(("control", "tutorial", "gamified")):
    v = by[g]["gain"]
    half = stats.t.ppf(0.975, len(v) - 1) * v.std(ddof=1) / math.sqrt(len(v))
    axes[0].errorbar(i, v.mean(), yerr=half, fmt="o", capsize=5, markersize=8)
axes[0].set_xticks(range(3)); axes[0].set_xticklabels(("control", "tutorial", "gamified")); axes[0].axhline(0, color="grey", linestyle=":")
axes[0].set_ylabel("Mean improvement (points)"); axes[0].set_title("Mean gain with 95% CI")

# (b) forest plot of standardised effects vs control
rows = []
for g in ("tutorial", "gamified"):
    rows.append((f"{g}: gain", *d_with_ci(by[g]["gain"], by["control"]["gain"])))
    rows.append((f"{g}: post score", *d_with_ci(by[g]["post_score"], by["control"]["post_score"])))
    rows.append((f"{g}: log minutes", *d_with_ci(np.log(by[g]["minutes_active"]), np.log(by["control"]["minutes_active"]))))
forest = pd.DataFrame(rows, columns=["effect", "d", "low", "high"])
ypos = np.arange(len(forest))[::-1]
axes[1].errorbar(forest["d"], ypos, xerr=[forest["d"] - forest["low"], forest["high"] - forest["d"]], fmt="s", capsize=4)
axes[1].axvline(0, color="grey", linestyle=":")
axes[1].set_yticks(ypos); axes[1].set_yticklabels(forest["effect"]); axes[1].set_xlabel("Cohen's d vs control (95% CI)")
axes[1].set_title("Forest plot: all effects on one scale")
plt.tight_layout(); plt.show()
forest.round(2)

In the forest plot an interval that **excludes 0** is a clear effect, and the width shows precision (it depends on the sample sizes and on $d$ itself). Using **gain scores** gives much *larger* standardised effects than raw post scores (gamified: $d\approx0.76$ vs $0.32$) because stable between-person differences have been removed from the noise. Tutorial shows a clear effect only on the gain, whereas gamified also differs on time in app. A single picture like this answers
"which effects are real, how big, and how sure are we?" far better than a table of $p$-values.

## 13. Choosing a test and writing it up

| design / outcome | test | effect size | assumptions to check |
|---|---|---|---|
| one mean vs a value | one-sample $t$ | $d$ | roughly normal / $n$ moderate |
| two independent means | **Welch $t$** | $d$ or $g$, mean difference + CI | independence; no extreme outliers |
| two independent, skewed/ordinal | Mann–Whitney $U$ | probability of superiority | independence |
| paired / before-after | **paired $t$** | $d_z$, mean change + CI | differences roughly normal |
| paired, skewed/ordinal | Wilcoxon signed-rank | rank-biserial / median change | symmetric differences |
| 3+ independent means | one-way ANOVA (+ Tukey/Holm) | $\eta^2$ | independence; similar spreads |
| 3+ independent, skewed/ordinal | Kruskal–Wallis | $\epsilon^2$ | independence |
| before/after with groups | **ANCOVA** or gain-score ANOVA | adjusted difference + CI | linear baseline relation |
| one proportion | exact binomial / Wilson CI | – | independent trials |
| two proportions | $z$-test or $\chi^2$; Fisher if small counts | difference, RR, OR | expected counts ≥ 5 |
| two categorical variables | $\chi^2$ independence | Cramér's $V$ | expected counts ≥ 5 |
| association of two numbers | Pearson / Spearman / Kendall | $r$, $\rho$, $\tau$ + CI | linearity / monotonicity |
| "no meaningful difference" | TOST equivalence | margin $\pm\Delta$ | margin chosen in advance |

One-page versions: [`reference/choosing-a-test.md`](../../reference/choosing-a-test.md) and [`reference/reporting-results.md`](../../reference/reporting-results.md).

**Reporting template (adapt to your test):**

> *Gamified onboarding improved scores by 4.4 points relative to control (95% CI 2.7 to 6.1; ANCOVA adjusting for pre-test score, n = 100 per group). Tutorial onboarding improved scores by 2.4 points (0.8 to 4.1).
> Pre-test scores were balanced across groups (equivalent within ±5 points, TOST p = 0.001). Comparisons were pre-specified; two treatment contrasts were tested, so Holm-adjusted p-values are reported in Table 2.*

## Recap

* Every test: statistic → null distribution → p-value; the **interval and effect size** carry the information.
* **Match the test to the design**: paired vs independent, 2 vs 3+ groups, numeric vs ordinal vs binary vs categorical.
* Skewed/ordinal → rank tests or logs; many comparisons → correct (or pre-specify); a baseline → ANCOVA; "no difference" → equivalence tests.
* **Plan power and sample size**; plot intervals and forest plots; run a **sensitivity analysis** instead of relying on pre-tests of assumptions.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (a hypothesis table).